In [2]:
import sys, os 
from pathlib import Path
ROOT = Path.cwd().parent
OFFICIAL = ROOT / "official" / "more-than-a-hack-2027"
sys.path.insert(0, str(OFFICIAL))
os.environ["MTAH_DATA_DIR"] = str(ROOT / "data" / "raw")
from games import asset_paths
paths = asset_paths()
paths 

{'video': '/Users/omarsherif/Documents/MTAH_2027/data/raw/tactical/BAR-ATH_tactical.mp4',
 'atd_meta': '/Users/omarsherif/Documents/MTAH_2027/data/raw/tactical/BAR-ATH_tactical_atd.xml',
 'atd_raw': '/Users/omarsherif/Documents/MTAH_2027/data/raw/tactical/BAR-ATH_tactical_atd.txt',
 'homography': '/Users/omarsherif/Documents/MTAH_2027/data/raw/tactical/BAR-ATH_tactical_homography.csv',
 'smart_tagging': '/Users/omarsherif/Documents/MTAH_2027/data/raw/tactical/BAR-ATH_tactical_smart_tagging.xml'}

In [3]:
for key, p in paths.items():
    p = Path(p)
    size = p.stat().st_size / 1e6 if p.exists() else None
    print(key, p.exists(), size)

video True 5222.307334
atd_meta True 0.059057
atd_raw True 195.714285
homography True 23.187043
smart_tagging True 0.261055


In [4]:
from kloppy import metrica

In [5]:
dataset = metrica.load_tracking_epts(
    meta_data=paths["atd_meta"],
    raw_data=paths["atd_raw"]
)

In [6]:
print(type(dataset))
print("number of frames:", len(dataset.frames))

<class 'kloppy.domain.models.tracking.TrackingDataset'>
number of frames: 199319


In [7]:
first_frame = dataset.frames[0]
print(type(first_frame))
print([
    name for name in dir(first_frame)
    if not name.startswith("_")
])

<class 'kloppy.domain.models.tracking.Frame'>
['attacking_direction', 'ball_coordinates', 'ball_owning_team', 'ball_speed', 'ball_state', 'dataset', 'frame_id', 'matches', 'next', 'next_record', 'other_data', 'period', 'players_coordinates', 'players_data', 'prev', 'prev_record', 'record_id', 'replace', 'set_refs', 'statistics', 'time', 'timestamp']


In [8]:
attributes_to_check = [
    "frame_id",
    "timestamp",
    "period",
    "ball_coordinates",
    "ball_speed",
    "ball_state",
    "ball_owning_team",
    "players_coordinates",
    "players_data"
]

for name in attributes_to_check:
    value = getattr(first_frame, name, None)

    print(
        name,
        "Type:", type(value).__name__,
        "Is None:", value is None
    )

frame_id Type: int Is None: False
timestamp Type: timedelta Is None: False
period Type: Period Is None: False
ball_coordinates Type: Point3D Is None: False
ball_speed Type: NoneType Is None: True
ball_state Type: NoneType Is None: True
ball_owning_team Type: NoneType Is None: True
players_coordinates Type: dict Is None: False
players_data Type: dict Is None: False


In [9]:
for index in [0, 1, 2, 100, 1000, 10000]:
    frame = dataset.frames[index]

    print(
        "Frame index:", index,
        "| frame_id:", frame.frame_id,
        "| ball_speed is None:", frame.ball_speed is None,
        "| ball_state is None:", frame.ball_state is None,
        "| ball_owning_team is None:", frame.ball_owning_team is None
    )

Frame index: 0 | frame_id: 1 | ball_speed is None: True | ball_state is None: True | ball_owning_team is None: True
Frame index: 1 | frame_id: 2 | ball_speed is None: True | ball_state is None: True | ball_owning_team is None: True
Frame index: 2 | frame_id: 3 | ball_speed is None: True | ball_state is None: True | ball_owning_team is None: True
Frame index: 100 | frame_id: 101 | ball_speed is None: True | ball_state is None: True | ball_owning_team is None: True
Frame index: 1000 | frame_id: 1001 | ball_speed is None: True | ball_state is None: True | ball_owning_team is None: True
Frame index: 10000 | frame_id: 10001 | ball_speed is None: True | ball_state is None: True | ball_owning_team is None: True


In [10]:
print("Frame type:", type(first_frame).__name__)

for name in ["ball_speed", "ball_state", "ball_owning_team"]:
    attribute = getattr(type(first_frame), name, None)

    print(
        name,
        "| Class attribute type:",
        type(attribute).__name__
    )

Frame type: Frame
ball_speed | Class attribute type: NoneType
ball_state | Class attribute type: NoneType
ball_owning_team | Class attribute type: NoneType


In [11]:
import inspect

print(inspect.getfile(type(first_frame)))

/Users/omarsherif/Documents/MTAH_2027/.venv/lib/python3.13/site-packages/kloppy/domain/models/tracking.py


In [12]:
frame_class = type(first_frame)
source = inspect.getsource(frame_class)

for line_number, line in enumerate(source.splitlines(), start=1):
    if any(
        field in line
        for field in [
            "ball_speed",
            "ball_state",
            "ball_owning_team",
            "ball_coordinates"
        ]
    ):
        print(f"Line {line_number}: {line.strip()}")

Line 9: ball_coordinates: The coordinates of the ball
Line 11: ball_speed: The speed of the ball
Line 18: ball_coordinates: Point3D
Line 19: ball_speed: Optional[float] = None


In [13]:
print(source)

@docstring_inherit_attributes(DataRecord)
@dataclass(repr=False)
class Frame(DataRecord):
    """
    Tracking data frame.

    Attributes:
        frame_id: The unique identifier of the frame. Aias for `record_id`.
        ball_coordinates: The coordinates of the ball
        players_data: A dictionary containing the tracking data for each player.
        ball_speed: The speed of the ball
        other_data: A dictionary containing additional data
    """

    frame_id: int
    players_data: Dict[Player, PlayerData]
    other_data: Dict[str, Any]
    ball_coordinates: Point3D
    ball_speed: Optional[float] = None

    @property
    def record_id(self) -> int:
        return self.frame_id

    @property
    def players_coordinates(self):
        return {
            player: player_data.coordinates
            for player, player_data in self.players_data.items()
        }

    def __str__(self):
        return f"<{self.__class__.__name__} frame_id='{self.frame_id}' time='{self.time}'>"

In [14]:
from dataclasses import fields

print("Parent classes:")

for cls in type(first_frame).__mro__:
    print(cls.__name__)

print("\nFrame fields:")

for field in fields(first_frame):
    print(field.name, "|", field.type)

Parent classes:
Frame
DataRecord
ABC
object

Frame fields:
dataset | Dataset
prev_record | Optional[Self]
next_record | Optional[Self]
period | Period
timestamp | timedelta
statistics | List[Statistic]
ball_owning_team | Optional[Team]
ball_state | Optional[BallState]
frame_id | <class 'int'>
players_data | typing.Dict[kloppy.domain.models.common.Player, kloppy.domain.models.tracking.PlayerData]
other_data | typing.Dict[str, typing.Any]
ball_coordinates | <class 'kloppy.domain.models.pitch.Point3D'>
ball_speed | typing.Optional[float]


In [15]:
data_record_class = type(first_frame).__mro__[1]
data_record_source = inspect.getsource(data_record_class)

print(data_record_source)

@dataclass
class DataRecord(ABC):
    """
    Base class for a data record in a dataset.

    Attributes:
        dataset: The dataset to which the record belongs.
        prev_record: The previous record in the dataset.
        next_record: The next record in the dataset.
        record_id: The unique identifier of the record. Given by the provider.
        period: The match period in which the observation occurred.
        timestamp: Timestamp of occurrence, relative to the period kick-off.
        time: The time of the observation. Combines `period` and `timestamp`.
        attacking_direction: The attacking direction of the home team.
        ball_owning_team: The team that had possession of the ball.
        ball_state: The state of the ball at the time of the observation.
    """

    dataset: Dataset = field(init=False)
    prev_record: Optional[Self] = field(init=False)
    next_record: Optional[Self] = field(init=False)
    period: Period
    timestamp: timedelta
    statistic

In [16]:
loader_source = inspect.getsource(metrica.load_tracking_epts)

print(loader_source)

def load_tracking_epts(
    meta_data: FileLike,
    raw_data: FileLike,
    sample_rate: Optional[float] = None,
    limit: Optional[int] = None,
    coordinates: Optional[str] = None,
) -> TrackingDataset:
    """
    Load Metrica Sports EPTS tracking data.

    Args:
        meta_data: A xml feed containing the match meta data.
        raw_data: A feed containing the raw tracking data in the EPTS format.
        sample_rate: Sample the data at a specific rate.
        limit: Limit the number of frames to load to the first `limit` frames.
        coordinates: The coordinate system to use.

    Returns:
        The parsed tracking data.
    """
    deserializer = MetricaEPTSTrackingDataDeserializer(
        sample_rate=sample_rate, limit=limit, coordinate_system=coordinates
    )
    with open_as_file(raw_data) as raw_data_fp, open_as_file(
        meta_data
    ) as meta_data_fp:
        return deserializer.deserialize(
            inputs=MetricaEPTSTrackingDataInputs(
              

In [17]:
loader_globals = metrica.load_tracking_epts.__globals__

deserializer_class = loader_globals[
    "MetricaEPTSTrackingDataDeserializer"
]

print("Class:", deserializer_class.__name__)
print("Defined in:", inspect.getfile(deserializer_class))

print("\nAvailable methods:")

for name, member in inspect.getmembers(deserializer_class):
    if inspect.isfunction(member):
        print(name)

Class: MetricaEPTSTrackingDataDeserializer
Defined in: /Users/omarsherif/Documents/MTAH_2027/.venv/lib/python3.13/site-packages/kloppy/infra/serializers/tracking/metrica_epts/deserializer.py

Available methods:
__init__
_frame_from_row
deserialize
get_transformer


In [18]:
import numpy as np

ids = np.array([f.frame_id for f in dataset.frames])
print("first id:", ids[0], "| last id:", ids[-1])
print("min step:", np.diff(ids).min(), "| max step:", np.diff(ids).max())

print("ball_state not None:", sum(f.ball_state is not None for f in dataset.frames))
print("ball_owning_team not None:", sum(f.ball_owning_team is not None for f in dataset.frames))

ball_x = np.array([f.ball_coordinates.x for f in dataset.frames], dtype=float)
print("ball missing share:", np.isnan(ball_x).mean())

first id: 1 | last id: 199319
min step: 1 | max step: 1
ball_state not None: 0
ball_owning_team not None: 0
ball missing share: 0.4306212654087167


In [19]:
import time
t0 = time.time()
df = dataset.to_df()
print("seconds:", round(time.time() - t0, 1))
print("shape:", df.shape)
print(list(df.columns[:12]))
print(type(df))


seconds: 5.8
shape: (199319, 193)
['period_id', 'timestamp', 'frame_id', 'ball_state', 'ball_owning_team_id', 'ball_x', 'ball_y', 'ball_z', 'ball_speed', '1_x', '1_y', '1_d']
<class 'pandas.DataFrame'>


In [20]:
print(list(df.columns[9:15]))
print(list(df.columns[-4:]))

print(df["period_id"].value_counts().sort_index())
print(df.groupby("period_id")["frame_id"].agg(["min", "max", "count"]))

print(df[["ball_state", "ball_owning_team_id"]].notna().sum())

['1_x', '1_y', '1_d', '1_s', '2_x', '2_y']
['46_x', '46_y', '46_d', '46_s']
period_id
1    199319
Name: count, dtype: int64
           min     max   count
period_id                     
1            1  199319  199319
ball_state             0
ball_owning_team_id    0
dtype: int64


In [33]:
import pandas as pd
pd.set_option("display.max_columns", 20)

In [35]:
df.head()

,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,1_x,...,44_d,44_s,45_x,45_y,45_d,45_s,46_x,46_y,46_d,46_s
0,1,0 days 00:00:00.040000,1,None,None,NaN,NaN,None,None,0.33411,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
1,1,0 days 00:00:00.080000,2,None,None,NaN,NaN,None,None,0.33404,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
2,1,0 days 00:00:00.120000,3,None,None,NaN,NaN,None,None,0.33396,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
3,1,0 days 00:00:00.160000,4,None,None,NaN,NaN,None,None,0.33389,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
4,1,0 days 00:00:00.200000,5,None,None,NaN,NaN,None,None,0.33382,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN


In [36]:
df.info(max_cols=15)

<class 'pandas.DataFrame'>
RangeIndex: 199319 entries, 0 to 199318
Columns: 193 entries, period_id to 46_s
dtypes: float64(140), int64(2), object(50), timedelta64[us](1)
memory usage: 293.5+ MB


In [37]:
df[["frame_id", "ball_x", "ball_y", "ball_z", "1_x", "1_y", "1_d", "1_s"]].head(10)

,frame_id,ball_x,ball_y,ball_z,1_x,1_y,1_d,1_s
0,1,NaN,NaN,None,0.33411,0.90565,None,0.886863
1,2,NaN,NaN,None,0.33404,0.90487,None,0.886863
2,3,NaN,NaN,None,0.33396,0.90413,None,0.886863
3,4,NaN,NaN,None,0.33389,0.90344,None,0.886863
4,5,NaN,NaN,None,0.33382,0.90279,None,0.886863
5,6,NaN,NaN,None,0.33375,0.90220,None,0.886863
6,7,NaN,NaN,None,0.33368,0.90167,None,0.886863
7,8,NaN,NaN,None,0.33362,0.90120,None,0.886863
8,9,NaN,NaN,None,0.33355,0.90080,None,0.886863
9,10,NaN,NaN,None,0.33349,0.90047,None,0.886863


In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 199319 entries, 0 to 199318
Columns: 193 entries, period_id to 46_s
dtypes: float64(140), int64(2), object(50), timedelta64[us](1)
memory usage: 293.5+ MB


In [30]:
meta = dataset.metadata
print([n for n in dir(meta) if not n.startswith("_")])
print(len(meta.teams))
for t in meta.teams:
    print(t.name, "|", t.ground, "|", len(t.players))

['attributes', 'away_coach', 'coordinate_system', 'data_format_specifications', 'date', 'flags', 'frame_rate', 'game_id', 'game_week', 'home_coach', 'officials', 'orientation', 'periods', 'pitch_dimensions', 'player_channels', 'provider', 'score', 'sensors', 'teams']
2
Barcelona | home | 24
Atletic Club | away | 22


In [39]:
print("ball_z non-null:", df["ball_z"].notna().sum())
print("ball_speed non-null:", df["ball_speed"].notna().sum())

d_cols = [c for c in df.columns if c.endswith("_d")]
s_cols = [c for c in df.columns if c.endswith("_s")]
print("d columns:", len(d_cols), "| non-null in all of them:", df[d_cols].notna().sum().sum())
print("s columns:", len(s_cols), "| non-null in all of them:", df[s_cols].notna().sum().sum())

print("ball_x min/max:", df["ball_x"].min(), df["ball_x"].max())
print("ball_y min/max:", df["ball_y"].min(), df["ball_y"].max())

ball_z non-null: 0
ball_speed non-null: 0
d columns: 46 | non-null in all of them: 0
s columns: 46 | non-null in all of them: 3412124
ball_x min/max: -0.06517 1.11385
ball_y min/max: -0.10713 1.48854


In [40]:
# frame video = index 
graded = np.zeros(len(df), dtype=bool)
graded[11185:90652] = True
graded[110808:190198] = True
print("graded frames:", graded.sum())

ball_ok = df["ball_x"].notna().to_numpy()
print("ball visible, all:", round(ball_ok.mean(), 4))
print("ball visible, graded:", round(ball_ok[graded].mean(), 4))

bx, by = df["ball_x"], df["ball_y"]
out = ((bx < 0) | (bx > 1) | (by < 0) | (by > 1)) & ball_ok
print("ball outside [0,1], share of visible:", round(out[ball_ok].mean(), 4))
far = ((bx < -0.05) | (bx > 1.05) | (by < -0.05) | (by > 1.05)) & ball_ok
print("ball far outside (>0.05), share of visible:", round(far[ball_ok].mean(), 4))

x_cols = [c for c in df.columns if c[0].isdigit() and c.endswith("_x")]
n_tracked = df[x_cols].notna().sum(axis=1).to_numpy()
print("tracked players per frame, graded: mean", round(n_tracked[graded].mean(), 2),
      "| min", n_tracked[graded].min(), "| max", n_tracked[graded].max())

graded frames: 158857
ball visible, all: 0.5694
ball visible, graded: 0.5931
ball outside [0,1], share of visible: 0.0437
ball far outside (>0.05), share of visible: 0.018
tracked players per frame, graded: mean 20.49 | min 1 | max 25


In [41]:
pc = meta.player_channels
print("channels:", len(pc), "| type:", type(pc[0]).__name__)
print([n for n in dir(pc[0]) if not n.startswith("_")])

p = meta.teams[0].players[0]
print(type(p).__name__)
print([n for n in dir(p) if not n.startswith("_")])

# قياس إضافي على الـ frames الغريبة (graded بس)
print("graded frames with <10 tracked:", (n_tracked[graded] < 10).sum())
print("graded frames with >22 tracked:", (n_tracked[graded] > 22).sum())

channels: 138 | type: PlayerChannel
['channel', 'player', 'player_channel_id']
Player
['attributes', 'first_name', 'full_name', 'jersey_no', 'last_name', 'name', 'player_id', 'position', 'positions', 'set_position', 'starting', 'starting_position', 'team']
graded frames with <10 tracked: 4838
graded frames with >22 tracked: 3749


In [43]:
STARTER = OFFICIAL / "starter"
sys.path.insert(0, str(STARTER))

from load_atd import check_alignment

alignment_report = check_alignment(dataset)

print("OFFICIAL ALIGNMENT REPORT")
for key, value in alignment_report.items():
    print(f"{key}: {value}")

assert alignment_report["agrees"], (
    "ATD alignment does not agree with the official timeline"
)


OFFICIAL ALIGNMENT REPORT
n_records: 199319
n_video_frames: 199319
frame_id_range: (1, 199319)
contiguous: True
first_video_frame: 0
agrees: True


In [44]:

from baseline import team_of_each_track, team_names

team_names_found = team_names(paths["atd_meta"])
track_teams = team_of_each_track(paths["atd_meta"])

print("Home:", team_names_found[0])
print("Away:", team_names_found[1])

print("Number of tracking slots:", len(track_teams))
print("Home slots:", int((track_teams == 0).sum()))
print("Away slots:", int((track_teams == 1).sum()))

assert len(track_teams) == 46
assert set(track_teams.tolist()) == {0, 1}


Home: Barcelona
Away: Atletic Club
Number of tracking slots: 46
Home slots: 24
Away slots: 22


In [45]:

from load_smart_tagging import load, check_time_base

events = load()

print("Total Smart Tagging events:", len(events))

time_report = check_time_base(events)

print("\nKickoff alignment:")
for item in time_report["halves"]:
    print(item)

print("\nTime base agrees:", time_report["agrees"])

import pandas as pd

tagging_summary = pd.DataFrame([
    {
        "code": event["code"],
        "start_frame": event["start_frame"],
        "end_frame": event["end_frame"],
        "team": event["tags"].get("Team"),
        "half": event["tags"].get("Half"),
        "type": event["tags"].get("Type")
    }
    for event in events
])

print("\nTagging codes:")
print(tagging_summary["code"].value_counts())

assert len(events) > 0
assert time_report["agrees"]


Total Smart Tagging events: 638

Kickoff alignment:
{'half': '1st Half', 'true_kickoff_frame': 11185, 'tagged': [(11097, 11222)], 'contains_kickoff': True}
{'half': '2nd Half', 'true_kickoff_frame': 113742, 'tagged': [(113660, 113785)], 'contains_kickoff': True}

Time base agrees: True

Tagging codes:
code
PLAYERS IN THE BOX              97
BALL IN FINAL THIRD             93
BALL IN THE BOX                 72
PROGRESSION                     66
DEFENDING IN MIDDLE THIRD       66
SET PIECES                      59
CREATING CHANCES                30
DEFENDING IN DEFENSIVE THIRD    30
LONG BALL                       29
ATTACKING TRANSITION            24
DEFENSIVE TRANSITION            24
BUILD UP                        22
DEFENDING IN ATTACKING THIRD    22
GOALS                            4
Name: count, dtype: int64


In [46]:

from load_homography import load as load_homography

homography_matrices, reprojection_error, homography_valid = (
    load_homography()
)

print("Matrices:", homography_matrices.shape)
print("Valid frames:", int(homography_valid.sum()))
print("Invalid frames:", int((~homography_valid).sum()))

print(
    "Median reprojection error:",
    float(pd.Series(reprojection_error).median())
)

assert homography_matrices.shape == (199319, 3, 3)
assert len(homography_valid) == 199319


Matrices: (199319, 3, 3)
Valid frames: 199319
Invalid frames: 0
Median reprojection error: 0.239245936


In [47]:

import json

summary = {
    "n_frames": len(df),
    "first_frame_id": int(ids[0]),
    "last_frame_id": int(ids[-1]),
    "home_team": team_names_found[0],
    "away_team": team_names_found[1],
    "n_tracking_slots": len(track_teams),
    "ball_visible_all": float(ball_ok.mean()),
    "ball_visible_graded": float(ball_ok[graded].mean()),
    "graded_frames": int(graded.sum()),
    "smart_tagging_events": len(events),
    "alignment_agrees": bool(alignment_report["agrees"]),
    "tagging_time_base_agrees": bool(time_report["agrees"]),
    "homography_valid_frames": int(homography_valid.sum())
}

with open(
    ROOT / "outputs" / "exploration_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("Exploration summary saved.")
print(summary)


Exploration summary saved.
{'n_frames': 199319, 'first_frame_id': 1, 'last_frame_id': 199319, 'home_team': 'Barcelona', 'away_team': 'Atletic Club', 'n_tracking_slots': 46, 'ball_visible_all': 0.5693787345912833, 'ball_visible_graded': 0.593074274347369, 'graded_frames': 158857, 'smart_tagging_events': 638, 'alignment_agrees': True, 'tagging_time_base_agrees': True, 'homography_valid_frames': 199319}


In [48]:
STARTER = OFFICIAL / "starter"

if str(STARTER) not in sys.path:
    sys.path.insert(0, str(STARTER))

from load_atd import check_alignment

alignment_report = check_alignment(dataset)

print("OFFICIAL ALIGNMENT REPORT")

for key, value in alignment_report.items():
    print(key, ":", value)

assert alignment_report["agrees"] is True

OFFICIAL ALIGNMENT REPORT
n_records : 199319
n_video_frames : 199319
frame_id_range : (1, 199319)
contiguous : True
first_video_frame : 0
agrees : True


In [49]:
from baseline import team_of_each_track, team_names

team_names_found = team_names(paths["atd_meta"])

track_teams = team_of_each_track(paths["atd_meta"])

print("Home team:", team_names_found[0])
print("Away team:", team_names_found[1])

print("Total tracking slots:", len(track_teams))
print("Home slots:", int((track_teams == 0).sum()))
print("Away slots:", int((track_teams == 1).sum()))

assert len(track_teams) == 46
assert set(track_teams.tolist()) == {0, 1}

Home team: Barcelona
Away team: Atletic Club
Total tracking slots: 46
Home slots: 24
Away slots: 22


In [50]:
from load_smart_tagging import load, check_time_base

events = load()

time_report = check_time_base(events)

print("Total events:", len(events))
print("Time base agrees:", time_report["agrees"])

for half in time_report["halves"]:
    print(half)

tagging_summary = pd.DataFrame([
    {
        "code": event["code"],
        "start_frame": event["start_frame"],
        "end_frame": event["end_frame"],
        "team": event["tags"].get("Team"),
        "half": event["tags"].get("Half"),
        "type": event["tags"].get("Type")
    }
    for event in events
])

print("\nEvent counts:")
print(tagging_summary["code"].value_counts())

print("\nTeam tags:")
print(tagging_summary["team"].value_counts())

assert len(events) == 638
assert time_report["agrees"]

Total events: 638
Time base agrees: True
{'half': '1st Half', 'true_kickoff_frame': 11185, 'tagged': [(11097, 11222)], 'contains_kickoff': True}
{'half': '2nd Half', 'true_kickoff_frame': 113742, 'tagged': [(113660, 113785)], 'contains_kickoff': True}

Event counts:
code
PLAYERS IN THE BOX              97
BALL IN FINAL THIRD             93
BALL IN THE BOX                 72
PROGRESSION                     66
DEFENDING IN MIDDLE THIRD       66
SET PIECES                      59
CREATING CHANCES                30
DEFENDING IN DEFENSIVE THIRD    30
LONG BALL                       29
ATTACKING TRANSITION            24
DEFENSIVE TRANSITION            24
BUILD UP                        22
DEFENDING IN ATTACKING THIRD    22
GOALS                            4
Name: count, dtype: int64

Team tags:
team
Barcelona       314
Atletic Club    288
N/A              36
Name: count, dtype: int64


In [51]:
from load_homography import load as load_homography

homography_matrices, reprojection_error, homography_valid = (
    load_homography()
)

print("Matrices:", homography_matrices.shape)
print("Valid frames:", homography_valid.sum())
print("Invalid frames:", (~homography_valid).sum())

print(
    "Median reprojection error:",
    np.nanmedian(reprojection_error)
)

assert homography_matrices.shape == (199319, 3, 3)
assert len(homography_valid) == 199319
assert homography_valid.all()

Matrices: (199319, 3, 3)
Valid frames: 199319
Invalid frames: 0
Median reprojection error: 0.239245936
